# TracIn - NLP Text Classification (DBPedia)

This notebook implements the TracIn algorithm for a text classification task on the DBPedia dataset, as described in **Section 5.2** of the paper _Estimating Training Data Influence by Tracing Gradient Descent_.

**CRITICAL UPDATE:** This version completely removes `torchtext`, uses Hugging Face `datasets`, and implements `torch.func.vmap` for extreme GPU acceleration over the full 560,000 example domain.

**CHECKPOINT STRATEGY:** Every expensive step saves to disk. Restarting and re-running will skip already-completed steps automatically.

In [ ]:
!pip install datasets tqdm

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import numpy as np
from tqdm.auto import tqdm
from torch.func import vmap, grad
import os
import pickle
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)

# ── Central checkpoint directory ──────────────────────────────────────────────
CKPT_DIR = "tracin_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Paths for every expensive artifact
VOCAB_PATH        = os.path.join(CKPT_DIR, "vocab.pkl")
MODEL_CKPT_PATH   = lambda ep: os.path.join(CKPT_DIR, f"model_epoch_{ep}.pt")
TRAIN_DONE_PATH   = os.path.join(CKPT_DIR, "training_done.json")   # tracks finished epochs
SCORES_PATH       = os.path.join(CKPT_DIR, "tracin_scores.npy")
SCORES_DONE_PATH  = os.path.join(CKPT_DIR, "scores_done.json")     # tracks finished ckpts

print(f"Checkpoint directory: {os.path.abspath(CKPT_DIR)}")
print("All saved artifacts:")
for f in sorted(os.listdir(CKPT_DIR)):
    print(f"  {f}")

Using device: cuda
Checkpoint directory: /kaggle/working/tracin_checkpoints
All saved artifacts:


In [5]:
print("Downloading DBPedia 14...")
dataset = load_dataset("dbpedia_14")

train_data = dataset["train"]
test_data  = dataset["test"]

print(f"Training examples : {len(train_data)}")
print(f"Test     examples : {len(test_data)}")

Training examples : 560000
Test     examples : 70000


In [6]:
def basic_tokenizer(text):
    return str(text).lower().split()

VOCAB_SIZE = 40000

if os.path.exists(VOCAB_PATH):
    print(f" Vocabulary already built — loading from {VOCAB_PATH}")
    with open(VOCAB_PATH, "rb") as f:
        vocab = pickle.load(f)
    print(f"Loaded vocabulary size: {len(vocab)}")
else:
    print("Building vocabulary from scratch (this is the slow part) ...")
    counter = Counter()
    for example in tqdm(train_data, desc="Building vocabulary"):
        counter.update(basic_tokenizer(example["content"]))

    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in counter.most_common(VOCAB_SIZE - 2):
        vocab[word] = len(vocab)

    with open(VOCAB_PATH, "wb") as f:
        pickle.dump(vocab, f)
    print(f" Vocabulary built and saved → {VOCAB_PATH}")
    print(f"Final Vocabulary Size: {len(vocab)}")

def encode_text(text):
    return [vocab.get(w, vocab["<unk>"]) for w in basic_tokenizer(text)]

Building vocabulary from scratch (this is the slow part) ...


Building vocabulary:   0%|          | 0/560000 [00:00<?, ?it/s]

 Vocabulary built and saved → tracin_checkpoints/vocab.pkl
Final Vocabulary Size: 40000


In [7]:
class DBPediaDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item["label"], item["content"]

train_pytorch_dataset = DBPediaDataset(train_data)
test_pytorch_dataset  = DBPediaDataset(test_data)

def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    for label, text in batch:
        label_list.append(label)
        encoded = encode_text(text)
        if len(encoded) == 0:
            encoded = [0]
        processed_text = torch.tensor(encoded, dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))

    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets    = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list  = torch.cat(text_list)
    return label_list, text_list, offsets

train_dataloader = DataLoader(
    train_pytorch_dataset, batch_size=256, shuffle=True,
    collate_fn=collate_batch, pin_memory=True
)
test_dataloader = DataLoader(
    test_pytorch_dataset, batch_size=256, shuffle=False,
    collate_fn=collate_batch, pin_memory=True
)
print("DataLoaders ready.")

DataLoaders ready.


In [8]:
embed_dim = 64
num_class = 14

class SWEM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(SWEM, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False, mode="mean")
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

model = SWEM(len(vocab), embed_dim, num_class).to(device)
print("Model architecture ready.")

Model architecture ready.


In [9]:
epochs            = 6
lr                = 1.0
criterion         = nn.CrossEntropyLoss()
optimizer         = torch.optim.SGD(model.parameters(), lr=lr)
checkpoint_epochs = [2, 4, 6]

# ── Load training-progress record ────────────────────────────────────────────
if os.path.exists(TRAIN_DONE_PATH):
    with open(TRAIN_DONE_PATH) as f:
        train_progress = json.load(f)       # {"completed_epochs": [2, 4]}
else:
    train_progress = {"completed_epochs": []}

# ── Resume: load latest finished checkpoint into model / optimizer ────────────
completed = train_progress["completed_epochs"]
if completed:
    latest_ep = max(completed)
    ckpt = torch.load(MODEL_CKPT_PATH(latest_ep), map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = latest_ep + 1
    print(f" Resumed from epoch {latest_ep}  — will continue from epoch {start_epoch}")
else:
    start_epoch = 1
    print("No previous training found — starting from epoch 1")

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, epochs + 1):
    model.train()
    total_loss, total_acc, total_count = 0, 0, 0

    for label, text, offsets in tqdm(train_dataloader, desc=f"Epoch {epoch}/{epochs}"):
        label, text, offsets = label.to(device), text.to(device), offsets.to(device)
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

        total_loss  += loss.item()
        total_acc   += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)

    avg_loss = total_loss / len(train_dataloader)
    avg_acc  = total_acc  / total_count
    print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

    # ── Save model after EVERY epoch (overwrites previous non-checkpoint save) ─
    every_epoch_path = os.path.join(CKPT_DIR, "model_latest.pt")
    torch.save({
        "epoch": epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "learning_rate":        lr,
        "loss":                 avg_loss,
        "acc":                  avg_acc,
    }, every_epoch_path)

    # ── Save named checkpoint at designated epochs ────────────────────────────
    if epoch in checkpoint_epochs:
        named_path = MODEL_CKPT_PATH(epoch)
        torch.save({
            "epoch": epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "learning_rate":        lr,
            "loss":                 avg_loss,
            "acc":                  avg_acc,
        }, named_path)
        print(f"   Named checkpoint saved → {named_path}")

    # ── Update progress record ────────────────────────────────────────────────
    if epoch not in train_progress["completed_epochs"]:
        train_progress["completed_epochs"].append(epoch)
    with open(TRAIN_DONE_PATH, "w") as f:
        json.dump(train_progress, f)

print("\n Training complete (or already was complete).")

# ── Collect checkpoint objects for TracIn ────────────────────────────────────
checkpoints = []
for ep in checkpoint_epochs:
    p = MODEL_CKPT_PATH(ep)
    if os.path.exists(p):
        ckpt = torch.load(p, map_location="cpu")
        checkpoints.append(ckpt)
        print(f"  Loaded TracIn checkpoint: epoch {ep}")
    else:
        print(f"  ⚠️  Missing checkpoint for epoch {ep} — TracIn will be incomplete")

# Ensure final model state is the fully trained one
final_ckpt = torch.load(MODEL_CKPT_PATH(epochs), map_location=device)
model.load_state_dict(final_ckpt["model_state_dict"])
print("Model loaded to its final trained state.")

No previous training found — starting from epoch 1


Epoch 1/6:   0%|          | 0/2188 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.9206 | Acc: 0.7784


Epoch 2/6:   0%|          | 0/2188 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.2925 | Acc: 0.9247
   Named checkpoint saved → tracin_checkpoints/model_epoch_2.pt


...

Epoch 4 | Loss: 0.1645 | Acc: 0.9572
   Named checkpoint saved → tracin_checkpoints/model_epoch_4.pt


Epoch 5/6:   0%|          | 0/2188 [00:00<?, ?it/s]

Epoch 5 | Loss: 0.1425 | Acc: 0.9628


Epoch 6/6:   0%|          | 0/2188 [00:00<?, ?it/s]

Epoch 6 | Loss: 0.1281 | Acc: 0.9665
   Named checkpoint saved → tracin_checkpoints/model_epoch_6.pt

 Training complete (or already was complete).
  Loaded TracIn checkpoint: epoch 2
  Loaded TracIn checkpoint: epoch 4
  Loaded TracIn checkpoint: epoch 6
Model loaded to its final trained state.


In [10]:
def compute_last_layer_gradient_nlp(model, text, offsets, label, criterion, device):
    model.zero_grad()
    output = model(text, offsets)
    loss = criterion(output, label)
    loss.backward()
    grad_weight = model.fc.weight.grad.view(-1)
    grad_bias   = model.fc.bias.grad.view(-1)
    return torch.cat([grad_weight, grad_bias]).detach().cpu()


def find_proponents_opponents_optimized(
    test_example, train_dataset, checkpoints, criterion, device,
    top_k=3, batch_size=512
):
    """
    GPU-optimised TracIn with per-checkpoint score accumulation checkpointing.
    Skips any checkpoint whose contribution has already been accumulated.
    """
    num_examples = len(train_dataset)

    # ── Load or initialise the scores array ──────────────────────────────────
    if os.path.exists(SCORES_PATH):
        scores = np.load(SCORES_PATH)
        print(f" Loaded partial scores from {SCORES_PATH}")
    else:
        scores = np.zeros(num_examples)

    # ── Load scores-progress record ──────────────────────────────────────────
    if os.path.exists(SCORES_DONE_PATH):
        with open(SCORES_DONE_PATH) as f:
            scores_progress = json.load(f)   # {"done_epochs": [2, 4]}
    else:
        scores_progress = {"done_epochs": []}

    # ── Prepare test gradient (needed for every checkpoint) ───────────────────
    test_label, test_text_str = test_example
    test_label_tensor, test_text_tensor, test_offsets = collate_batch(
        [(test_label, test_text_str)]
    )
    test_label_tensor = test_label_tensor.to(device)
    test_text_tensor  = test_text_tensor.to(device)
    test_offsets      = test_offsets.to(device)

    dataloader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_batch, pin_memory=True
    )

    for checkpoint in checkpoints:
        ep = checkpoint["epoch"]

        if ep in scores_progress["done_epochs"]:
            print(f" Epoch {ep} scores already accumulated — skipping")
            continue

        print(f"Computing TracIn contributions from epoch {ep} checkpoint ...")

        ckpt_model = SWEM(len(vocab), embed_dim, num_class).to(device)
        ckpt_model.load_state_dict(
            {k: v.to(device) for k, v in checkpoint["model_state_dict"].items()}
        )
        ckpt_model.eval()
        ckpt_lr = checkpoint["learning_rate"]

        grad_test = compute_last_layer_gradient_nlp(
            ckpt_model, test_text_tensor, test_offsets,
            test_label_tensor, criterion, device
        ).to(device)

        params  = {k: v.detach() for k, v in ckpt_model.fc.named_parameters()}
        buffers = {}

        def compute_loss_fc_only(params, buffers, features, target):
            logits = torch.nn.functional.linear(
                features, params['weight'], params.get('bias')
            )
            return criterion(logits.unsqueeze(0), target.unsqueeze(0))

        ft_compute_grad        = grad(compute_loss_fc_only)
        ft_compute_sample_grad = vmap(ft_compute_grad, in_dims=(None, None, 0, 0))

        current_idx = 0
        for batch_labels, batch_text, batch_offsets in tqdm(
            dataloader, desc=f"Tracing Epoch {ep}"
        ):
            batch_labels  = batch_labels.to(device)
            batch_text    = batch_text.to(device)
            batch_offsets = batch_offsets.to(device)

            with torch.no_grad():
                features = ckpt_model.embedding(batch_text, batch_offsets)

            per_sample_grads = ft_compute_sample_grad(
                params, buffers, features, batch_labels
            )

            bias_grads = (
                per_sample_grads['bias']
                if 'bias' in per_sample_grads
                else torch.zeros(len(batch_labels), num_class, device=device)
            )

            # Vectorised dot-product: (N, d) @ (d,) → (N,)
            w_flat = per_sample_grads['weight'].flatten(1)   # (N, d_w)
            b_flat = bias_grads.flatten(1)                   # (N, d_b)
            g_flat = torch.cat([w_flat, b_flat], dim=1)      # (N, d_w+d_b)
            dots   = (g_flat * grad_test.unsqueeze(0)).sum(1) # (N,)

            batch_sz = len(batch_labels)
            scores[current_idx : current_idx + batch_sz] += (
                ckpt_lr * dots.cpu().numpy()
            )
            current_idx += batch_sz

        # ── Save after each checkpoint ───────────────────────────────────────
        np.save(SCORES_PATH, scores)
        scores_progress["done_epochs"].append(ep)
        with open(SCORES_DONE_PATH, "w") as f:
            json.dump(scores_progress, f)
        print(f"   Scores saved after epoch {ep} checkpoint")

    indexed_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    proponents = [(s, i) for i, s in indexed_scores[:top_k]]
    opponents  = [(s, i) for i, s in indexed_scores[-top_k:]]
    return proponents, opponents

In [11]:
label_names = [
    "Company", "EducationalInstitution", "Artist", "Athlete", "OfficeHolder",
    "MeanOfTransportation", "Building", "NaturalPlace", "Village", "Animal",
    "Plant", "Album", "Film", "WrittenWork"
]

test_idx     = 10
test_example = test_pytorch_dataset[test_idx]
test_label_id, test_text = test_example

print(f"--- TEST EXAMPLE ---")
print(f"Class : {label_names[test_label_id]} ({test_label_id})")
print(f"Text  : {test_text[:300]}...\n")

proponents, opponents = find_proponents_opponents_optimized(
    test_example, train_pytorch_dataset, checkpoints,
    criterion, device, top_k=3, batch_size=512
)

print("\n=== TOP 3 PROPONENTS (Helpful Examples) ===")
for score, idx in proponents:
    label, text = train_pytorch_dataset[idx]
    print(f"Score: {score:.4f} | Class: {label_names[label]} ({label})")
    print(f"Text : {text[:200]}...\n")

print("\n=== TOP 3 OPPONENTS (Harmful Examples) ===")
for score, idx in opponents[::-1]:
    label, text = train_pytorch_dataset[idx]
    print(f"Score: {score:.4f} | Class: {label_names[label]} ({label})")
    print(f"Text : {text[:200]}...\n")

--- TEST EXAMPLE ---
Class : Company (0)
Text  :  MEPC plc is a leading British-based property investment and development business. It is headquartered in London. It used to be listed on the London Stock Exchange and was once a constituent of the FTSE 100 Index. It is however now owned by the British Telecom Pension Fund and the Royal Mail Pension...

Computing TracIn contributions from epoch 2 checkpoint ...


Tracing Epoch 2:   0%|          | 0/1094 [00:00<?, ?it/s]

   Scores saved after epoch 2 checkpoint
Computing TracIn contributions from epoch 4 checkpoint ...


Tracing Epoch 4:   0%|          | 0/1094 [00:00<?, ?it/s]

   Scores saved after epoch 4 checkpoint
Computing TracIn contributions from epoch 6 checkpoint ...


Tracing Epoch 6:   0%|          | 0/1094 [00:00<?, ?it/s]

   Scores saved after epoch 6 checkpoint

=== TOP 3 PROPONENTS (Helpful Examples) ===
Score: 1.1675 | Class: Company (0)
Text :  Diardi is a sports car manufactured in Turkey by the Çenberci Group....

Score: 1.1542 | Class: Company (0)
Text :  Shincho (新潮 Shinchō) is a Japanese literary magazine published monthly by Shinchosha. Since its launch in 1904 it has published the works of many of Japan's leading writers....

Score: 1.1442 | Class: Company (0)
Text :  StarWraith is a series of space combat simulators by StarWraith 3D Games....


=== TOP 3 OPPONENTS (Harmful Examples) ===
Score: -1.1560 | Class: MeanOfTransportation (5)
Text :  The Scaled Composites Model 339 SpaceShipTwo (SS2) is a suborbital air-launched spaceplane designed for space tourism. It is under development as part of the Tier 1b program under contract to The Spa...

Score: -1.1262 | Class: MeanOfTransportation (5)
Text :  Chalmers Motor Car Company was a United States based automobile company located in Detroit Mic